# 🔤 Teach a Model a *Lipogram* — GRPO + reward design

Train **Qwen3.5-2B** to answer questions **without ever using the letter “e”** — a *lipogram*.
Inspired by the [Thinking Machines “Inkling” demo](https://thinkingmachines.ai/news/introducing-inkling/).

Why this makes a great RL tutorial: the constraint is **verifiable** (just check for “e”), so the
reward is easy to write — and that's exactly why it's the perfect place to learn **reward design** and
the classic **reward-hacking** failure mode. A lazy reward gets gamed (the model spits out short,
letter-free gibberish). We'll *watch* that happen, then fix it.

Unlike notebook 01 (which used an OpenEnv server for the reward), here the reward is a plain **Python
function you own** — the other way rewards enter GRPO. Only a **Hugging Face token** is needed.

## 1. Install

In [ ]:
%pip install -q -U "trl>=0.19" datasets accelerate trackio 2>/dev/null
print('✅ installed')

## 🔑 Log in to Hugging Face

In [ ]:
from huggingface_hub import notebook_login, whoami
try:
    who = whoami()  # already logged in (HF_TOKEN in the job)
except Exception:
    notebook_login()
    who = whoami()
print('👤 logged in as:', who['name'])

## 2. Settings

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
USERNAME          = whoami()['name']

MODEL             = 'Qwen/Qwen3.5-2B'      # small text model; any chat LLM works
FORBIDDEN_LETTER  = 'e'                     # the letter the model must avoid
DATASET           = 'yahma/alpaca-cleaned'  # open-ended instructions -> avoiding 'e' is genuinely hard

NUM_TRAIN_SAMPLES = 2000
MAX_STEPS         = 40                       # short demo; bump to 300+ to really move it
NUM_GENERATIONS   = 8
MAX_COMPLETION    = 256
LEARNING_RATE     = 1e-5
REWARD_K          = 4.0                      # steepness of the letter penalty (higher = harsher)
MIN_WORDS         = 6                        # anti-hack: answers below this score 0

USE_TRACKIO       = True
TRACKIO_PROJECT   = 'lipogram-grpo'
TRACKIO_SPACE_ID  = f'{USERNAME}/trackio-lipogram'
print(f'🧠 {MODEL}  ·  forbid "{FORBIDDEN_LETTER}"  ·  {MAX_STEPS} steps × {NUM_GENERATIONS} gens')

## 3. The reward — and how it gets *hacked* 🎯

**This is the whole point of the notebook.** The obvious reward is “penalize the letter e.” But if that's
*all* you reward, the model discovers it can win by producing **short, repetitive, letter-free gibberish**
— maximal reward, zero usefulness. That's **reward hacking**.

So our reward is a **product of three terms**, so all must hold:

```
reward = lipogram_score  ×  length_guard  ×  diversity_guard
```
- **lipogram_score** — steep `exp(-k · e_per_word)`: 1.0 with zero “e”, dropping fast per “e”.
- **length_guard** — 0 below `MIN_WORDS` (kills 1-word hacks), decays for rambling.
- **diversity_guard** — penalizes low unique-word ratio (kills “no no no no”).

Run the cell — it scores a few hand-written answers so you can *see* the naive-vs-guarded difference.

In [ ]:
import math
L = FORBIDDEN_LETTER.lower()

def naive_reward(text):
    """The lazy reward: just 1 - fraction of words containing the letter. Gets hacked."""
    w = text.split();  wc = max(len(w), 1)
    return 1.0 - sum(1 for x in w if L in x.lower()) / wc

def guarded_reward(text, k=REWARD_K, min_words=MIN_WORDS, max_words=120, div_floor=0.4):
    """Composite reward: avoid the letter AND stay a real, varied answer."""
    w = text.split();  wc = len(w)
    if wc < min_words:              # empty / one-word letter-free hack
        return 0.0
    n = text.lower().count(L)
    lipo   = math.exp(-k * (n / wc))               # 1.0 at zero 'e', steep penalty per 'e'
    length = 1.0 if wc <= max_words else max(0.0, 1 - (wc - max_words) / max_words)
    div    = min(1.0, (len(set(x.lower() for x in w)) / wc) / div_floor)
    return lipo * length * div

examples = {
    'normal answer (lots of e)': 'The weather here is really pleasant and the trees are green everywhere.',
    'HACK: one word, no e'     : 'Ok.',
    'HACK: repetition, no e'   : 'blah blah blah blah blah blah blah blah',
    'GOOD: real answer, no e'  : 'My dog ran fast across a wide grassy hill on a warm sunny day, glad and calm.',
}
print(f"{'completion':30s} {'naive':>7} {'guarded':>8}")
for name, t in examples.items():
    print(f'{name:30s} {naive_reward(t):7.2f} {guarded_reward(t):8.2f}')
print('\n👆 naive rewards the gibberish hacks ~1.0; guarded gives them ~0 and only the real e-free answer wins.')

### The GRPO reward function
GRPO calls a reward function with a batch of `completions` and returns one score each. We wrap the
guarded reward above.

In [ ]:
SUFFIX = (f"\n\nRules: (1) never use the letter '{FORBIDDEN_LETTER}' anywhere in your reply; "
          f"(2) keep it brief and to the point.")

def lipogram_reward(completions, **kwargs):
    return [guarded_reward((c[0]['content'] or '').strip()) for c in completions]

## 4. The dataset
Open-ended alpaca instructions, each with the “don't use e” rule appended. GRPO only needs the
**prompt** (the reward scores the model's own answer — there's no gold label to match).

In [ ]:
from datasets import load_dataset, Dataset
raw = load_dataset(DATASET, split=f'train[:{NUM_TRAIN_SAMPLES}]')
prompts = []
for r in raw:
    q = (r.get('instruction') or '').strip()
    inp = (r.get('input') or '').strip()
    if inp: q = f'{q}\n\n{inp}'
    if q and len(q) <= 600:
        prompts.append([{'role':'user','content': q + SUFFIX}])
train_dataset = Dataset.from_dict({'prompt': prompts})
EVAL_QUESTIONS = [
    'What are the benefits of regular exercise?',
    'Explain how photosynthesis works.',
    'Give three tips for staying productive.',
    'Describe your favorite season and why you like it.',
    'How do airplanes stay in the air?',
]
print(f'📚 {len(train_dataset)} training prompts')

## 5. Baseline — how often is it e-free *now*?
Spoiler: almost never. Normal English is saturated with “e”.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
def load_model():
    return AutoModelForCausalLM.from_pretrained(MODEL, dtype='bfloat16', device_map='cuda')

def evaluate(model, questions=EVAL_QUESTIONS, show=True):
    model.eval(); ok = 0; samples = []
    for q in questions:
        msgs = [{'role':'user','content': q + SUFFIX}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        ins = tok(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**ins, max_new_tokens=MAX_COMPLETION, do_sample=False)
        ans = tok.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        e = ans.lower().count(L); ok += (e == 0 and len(ans.split()) >= MIN_WORDS)
        samples.append((q, ans, e))
    rate = ok / len(questions)
    if show:
        print(f'lipogram success: {rate*100:.0f}%  (answers with ZERO "{FORBIDDEN_LETTER}")')
        for q, a, e in samples[:2]: print(f'  Q: {q}\n  A: {a[:160]}  [e={e}]\n')
    return rate

base_model = load_model()
print('=== BEFORE training ==='); baseline = evaluate(base_model)
del base_model; torch.cuda.empty_cache()

## 6. Train 🏋️
`GRPOTrainer` samples `NUM_GENERATIONS` answers per prompt, scores them with `lipogram_reward`, and
nudges the model toward the higher-reward (e-free, real) ones. Short demo run.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
cfg = GRPOConfig(
    output_dir='lipogram-grpo',
    model_init_kwargs={'dtype':'bfloat16'},
    learning_rate=LEARNING_RATE,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=NUM_GENERATIONS,
    steps_per_generation=1, gradient_accumulation_steps=1,
    max_steps=MAX_STEPS, max_completion_length=MAX_COMPLETION,
    temperature=1.0,
    chat_template_kwargs={'enable_thinking': False},
    bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=1, save_strategy='no',
    report_to=('trackio' if USE_TRACKIO else 'none'),
    project=TRACKIO_PROJECT, trackio_space_id=(TRACKIO_SPACE_ID if USE_TRACKIO else None),
    run_name='lipogram-grpo',
)
trainer = GRPOTrainer(model=MODEL, args=cfg, train_dataset=train_dataset, reward_funcs=lipogram_reward)
trainer.train()

## 7. Did it learn to avoid “e”?

In [ ]:
print('=== AFTER training ==='); after = evaluate(trainer.model)
print(f'\nlipogram success:  {baseline*100:.0f}%  →  {after*100:.0f}%')

## Recap
- An RL **reward is just a function** of the model's output — here, a Python function you fully control.
- The **naive** reward was hackable (gibberish scored ~1.0); the **guarded** reward (letter × length ×
  diversity) made the model produce *real, e-free* answers.
- Same GRPO machinery as notebook 01 — the only difference is **where the reward comes from**
  (an OpenEnv server there, an in-process function here).
- 40 steps is a demo; a real lipogram model needs a few hundred steps + a stronger LR.

**Takeaway for your own envs: the reward is the whole game.** Design it so the *only* way to score high
is to actually do the task.